# 7.3 — Regression Evaluators

**Chapter 7, section 7.3.2**, and the assembler listing of **section 7.4**. The starting point
for **Exercise 8**.

**The question this notebook answers:** why does a comparison between two models need more than
one number?

An `Evaluator` turns a DataFrame of predictions into a single score, and it computes **exactly
one metric**, fixed when the evaluator is constructed. Comparing on two metrics therefore means
two evaluator objects. The notebook fits two models on the same features, scores each with the
same pair of evaluators, and finds that the two metrics disagree about how much better the
second model is: one says five percent, the other says a factor of seven.

Source: `Notebooks old/Spark-Example-21-Mllib-Regression.ipynb`, modernized and revised.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.3")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The data

Sixty-five countries, each with the size of the local Netflix catalogue and the price of the
three subscription tiers. The task is to predict the premium monthly price from the size of the
library.

It is a small and rather hopeless regression problem, and that is useful here: a model that
explains almost nothing is the clearest way to see two metrics describe the same failure
differently.

In [2]:
data = spark.read.csv(os.path.join(DATA, "netflix-subscription.csv"),
                      header=True, inferSchema=True)
print("rows:", data.count())
data.select("Country", "Total_Library_Size", "Num_TV_Shows", "Num_Movies",
            "Cost_Per_Month_Premium").show(5)

rows: 65
+---------+------------------+------------+----------+----------------------+
|  Country|Total_Library_Size|Num_TV_Shows|Num_Movies|Cost_Per_Month_Premium|
+---------+------------------+------------+----------+----------------------+
|Argentina|              4760|        3154|      1606|                  9.26|
|Australia|              6114|        4050|      2064|                 16.39|
|  Austria|              5640|        3779|      1861|                 20.32|
|  Belgium|              4990|        3374|      1616|                 20.32|
|  Bolivia|              4991|        3155|      1836|                 13.99|
+---------+------------------+------------+----------+----------------------+
only showing top 5 rows


---

## 2. Three columns into one vector

`VectorAssembler` concatenates the input columns **in the order given**. That order is not
cosmetic: it is the order in which the fitted model's coefficients come back, and there is
nothing in the coefficient vector to tell you which column each one belongs to.

In [3]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["Total_Library_Size", "Num_TV_Shows", "Num_Movies"],
    outputCol="features")

data = assembler.transform(data)
data.select("Total_Library_Size", "Num_TV_Shows", "Num_Movies", "features").show(5,
                                                                                 truncate=False)

+------------------+------------+----------+----------------------+
|Total_Library_Size|Num_TV_Shows|Num_Movies|features              |
+------------------+------------+----------+----------------------+
|4760              |3154        |1606      |[4760.0,3154.0,1606.0]|
|6114              |4050        |2064      |[6114.0,4050.0,2064.0]|
|5640              |3779        |1861      |[5640.0,3779.0,1861.0]|
|4990              |3374        |1616      |[4990.0,3374.0,1616.0]|
|4991              |3155        |1836      |[4991.0,3155.0,1836.0]|
+------------------+------------+----------+----------------------+
only showing top 5 rows


In [4]:
train_data, test_data = data.randomSplit([0.7, 0.3], seed=10)
print("training:", train_data.count(), " test:", test_data.count())

training: 45  test: 20


---

## 3. A linear model

In [5]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(labelCol="Cost_Per_Month_Premium", maxIter=10, regParam=0.1)
lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

print("Coefficients:", lr_model.coefficients)
print("Intercept:   ", lr_model.intercept)

Coefficients: [4.1810614493165303e-05,0.0005929975095584922,-0.0028238464511083665]
Intercept:    18.03030820681109


In [6]:
# Which coefficient belongs to which column: the assembler's input order, and nothing else.
for name, coef in zip(assembler.getInputCols(), lr_model.coefficients):
    print(f"  {name:<20} {coef: .6f}")

  Total_Library_Size    0.000042
  Num_TV_Shows          0.000593
  Num_Movies           -0.002824


In [7]:
lr_predictions.select("Country", "Cost_Per_Month_Premium", "prediction").show(5)

+-----------+----------------------+------------------+
|    Country|Cost_Per_Month_Premium|        prediction|
+-----------+----------------------+------------------+
|    Austria|                 20.32|15.251879415661413|
|    Belgium|                 20.32|15.676380905391216|
|     Brazil|                  9.96|15.002086630788916|
|     Canada|                 15.03|15.403200936603676|
|Switzerland|                 26.96|15.197566722684494|
+-----------+----------------------+------------------+
only showing top 5 rows


---

## 4. Two evaluators, because one evaluator is one metric

`metricName` is set when the object is built. Asking the same evaluator for a second metric is
not possible; a second metric is a second object and a second pass over the predictions.

In [8]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator1 = RegressionEvaluator(labelCol="Cost_Per_Month_Premium",
                                 predictionCol="prediction", metricName="rmse")
evaluator2 = RegressionEvaluator(labelCol="Cost_Per_Month_Premium",
                                 predictionCol="prediction", metricName="r2")

lr_rmse = evaluator1.evaluate(lr_predictions)
lr_r2 = evaluator2.evaluate(lr_predictions)
print(f"LinearRegression   RMSE = {lr_rmse:.4f}   R2 = {lr_r2:.4f}")

LinearRegression   RMSE = 4.2392   R2 = 0.0148


---

## 5. A non-linear model on the same features

Nothing about the featurization changes. The same assembled vectors, the same split, the same
two evaluators — only the estimator is different.

In [9]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(featuresCol="features", labelCol="Cost_Per_Month_Premium",
                   maxIter=3, maxDepth=2, seed=0)
gbt_model = gbt.fit(train_data)
gbt_predictions = gbt_model.transform(test_data)

gbt_rmse = evaluator1.evaluate(gbt_predictions)
gbt_r2 = evaluator2.evaluate(gbt_predictions)
print(f"GBTRegressor       RMSE = {gbt_rmse:.4f}   R2 = {gbt_r2:.4f}")

GBTRegressor       RMSE = 4.0328   R2 = 0.1084


---

## 6. The two metrics disagree about the size of the improvement

In [10]:
print(f"{'model':<20}{'RMSE':>10}{'R2':>10}")
print(f"{'LinearRegression':<20}{lr_rmse:>10.4f}{lr_r2:>10.4f}")
print(f"{'GBTRegressor':<20}{gbt_rmse:>10.4f}{gbt_r2:>10.4f}")
print()
print(f"RMSE improves by {100 * (lr_rmse - gbt_rmse) / lr_rmse:.1f}%")
print(f"R2   improves by a factor of {gbt_r2 / lr_r2:.1f}")

model                     RMSE        R2
LinearRegression        4.2392    0.0148
GBTRegressor            4.0328    0.1084

RMSE improves by 4.9%
R2   improves by a factor of 7.3


The same pair of models, the same predictions, and two readings that are not close to each
other. Both are correct, because they measure different things.

* **RMSE** is the typical size of a prediction error, in dollars per month. Going from 4.24 to
  4.03 is a five percent reduction in that error, and it is small because most of the error was
  never going to go away.
* **$R^2$** is the fraction of the variance in the target that the model accounts for. Going
  from 0.015 to 0.108 is a sevenfold increase in that fraction, and it is large because the
  starting point was almost zero.

Neither model is any good. The point is not which one wins but that a single number, chosen in
advance and reported alone, would have described this comparison as either a rounding error or a
breakthrough.

In [11]:
# Where the RMSE actually comes from: the mean of the squared residuals, square-rooted.
# Computing it by hand costs three lines and confirms what the evaluator returned.
import pyspark.sql.functions as F

by_hand = (gbt_predictions
           .select(F.sqrt(F.avg((F.col("Cost_Per_Month_Premium") - F.col("prediction")) ** 2))
                   .alias("rmse"))
           .first()["rmse"])

assert abs(by_hand - gbt_rmse) < 1e-9
print(f"RMSE by hand {by_hand:.6f}  ==  evaluator {gbt_rmse:.6f}")

RMSE by hand 4.032766  ==  evaluator 4.032766


---

## 7. Why this matters for a tuning search

A `CrossValidator` takes one evaluator, and it optimizes whatever that evaluator returns. Had
this comparison been a grid search rather than two hand-fitted models, the choice of
`metricName` at construction time would have chosen the winner — and the two metrics above rank
the improvement so differently that a search told to care about RMSE and a search told to care
about $R^2$ would report very different verdicts about whether the extra model complexity was
worth anything.

The evaluator is not a reporting decision made after the modelling. It **is** the modelling
objective.

In [12]:
# The four metrics a RegressionEvaluator can produce, from the same predictions.
for metric in ("rmse", "mse", "mae", "r2"):
    value = RegressionEvaluator(labelCol="Cost_Per_Month_Premium",
                                predictionCol="prediction",
                                metricName=metric).evaluate(gbt_predictions)
    print(f"  {metric:<5} {value:10.4f}")

  rmse      4.0328


  mse      16.2632


  mae       3.3961
  r2        0.1084


---

## What to take away

An evaluator computes one metric, fixed at construction. Two metrics mean two objects.

On this data and this split — sixty-five countries, seed 10, forty-five training rows and twenty
test rows — the linear model scores **RMSE 4.24 and $R^2$ 0.015**, and the gradient-boosted tree
scores **RMSE 4.03 and $R^2$ 0.108**. The first number improves by five percent and the second
by a factor of seven, from the same pair of prediction sets.

In [13]:
spark.stop()